# Serrodyne Waveform Example

Generate a piecewise serrodyne sawtooth waveform with `firmware.signals.serrodyne`, tile it to the DAC BRAM length, preview the expected EOM spectrum, and load it into DAC0.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from firmware import OverlayController
from firmware.signals import serrodyne

In [ ]:
ol = OverlayController()
info = ol.info()

DAC_SR = float(info["rfdc"]["dac0_sampling_rate_gsps"]) * 1e9
BUF_LEN = int(info["dac0"]["bram_int16_samples"])
DAC_PEAK = int(0.8 * np.iinfo(np.int16).max)

DAC_SR, BUF_LEN, DAC_PEAK

In [ ]:
def to_int16(waveform, peak=DAC_PEAK):
    data = np.asarray(waveform, dtype=float)
    return np.round(np.clip(data, -float(peak), float(peak))).astype(np.int16)


def tile_to_length(x, y, length):
    reps = int(np.ceil(length / len(y)))
    return np.tile(x, reps)[:length], np.tile(y, reps)[:length]


def plot_time(waveform, sample_rate, samples=4096, title="Waveform"):
    view = np.asarray(waveform)[:samples]
    time_ns = np.arange(view.size) / sample_rate * 1e9
    fig, ax = plt.subplots(figsize=(11, 3))
    ax.plot(time_ns, view)
    ax.set_title(title)
    ax.set_xlabel("Time (ns)")
    ax.set_ylabel("DAC code")
    ax.grid(True, alpha=0.3)
    return ax


def plot_eom_spectrum(phase_codes, sample_rate, title="Expected EOM spectrum"):
    data = np.asarray(phase_codes, dtype=float)
    span = float(np.max(data) - np.min(data))
    if span == 0.0:
        raise ValueError("Cannot plot spectrum for a constant waveform")
    phase = 2 * np.pi * data / span
    field = np.exp(1j * phase)
    spectrum = np.fft.fftshift(np.abs(np.fft.fft(field))) / len(field)
    freqs_mhz = np.fft.fftshift(np.fft.fftfreq(len(field), d=1 / sample_rate)) / 1e6

    fig, ax = plt.subplots(figsize=(11, 3))
    ax.plot(freqs_mhz, spectrum)
    ax.set_title(title)
    ax.set_xlabel("Frequency (MHz)")
    ax.set_ylabel("Magnitude")
    ax.grid(True, alpha=0.3)
    return ax

## Generate Serrodyne Pattern

In [ ]:
ratios = [1, 5, 3]
freqs_hz = [-1330e6, 0.0, 840e6]
total_seconds = 1.0e-6

x_base, y_base, n_base = serrodyne(
    ratios,
    freqs_hz,
    total_seconds,
    amplitude=DAC_PEAK,
    sample_rate=DAC_SR,
    max_points=BUF_LEN,
    continuous_phase=False,
)

_, y_full = tile_to_length(x_base, y_base, BUF_LEN)
dac0_waveform = to_int16(y_full)

n_base, dac0_waveform.dtype, dac0_waveform.shape

In [ ]:
plot_time(dac0_waveform, DAC_SR, title="DAC0 serrodyne waveform")
plot_eom_spectrum(dac0_waveform, DAC_SR, title="Expected serrodyne EOM spectrum");

## Program DAC0

In [ ]:
ol.dac0.load_waveform(dac0_waveform)
ol.info()

## Enable and Disable DAC0

In [ ]:
ol.dac0.enable()
ol.dac0.is_enabled()

In [ ]:
ol.dac0.disable()
ol.info()